# Track B Architecture Comparison

这个 notebook 用来做 Track B 架构实验台。当前已经接入 `encoder_only` Transformer、`pure_rnn` 和 `rnn_lstm_hybrid`，后面你可以把别的架构 runner 按同样接口注册进来做横向对比。

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from encoder_only_transformer import (
    ClusteringConfig,
    EncoderOnlyTransformerConfig,
    HMMLEARN_AVAILABLE,
    HMMReferenceConfig,
    SequenceDataConfig,
    TrainingConfig,
    compare_experiment_summaries,
    run_encoder_only_experiment,
)
from rnn_baselines import (
    RecurrentBaselineConfig,
    run_pure_rnn_experiment,
    run_rnn_lstm_hybrid_experiment,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("HMM reference available:", HMMLEARN_AVAILABLE)
if not HMMLEARN_AVAILABLE:
    print("`hmmlearn` 未安装，当前 notebook 会默认关闭 HMM 对照，只跑 encoder-only Transformer + clustering。")


## Single Experiment Configuration

先在这里配一个单独实验。当前支持三种架构：`encoder_only`、`pure_rnn`、`rnn_lstm_hybrid`。后面如果你接入新的架构，只要把新的 runner 注册进 `ARCHITECTURE_RUNNERS` 就可以复用下面的流程。

In [ ]:
ARCHITECTURE_RUNNERS = {
    "encoder_only": run_encoder_only_experiment,
    "pure_rnn": run_pure_rnn_experiment,
    "rnn_lstm_hybrid": run_rnn_lstm_hybrid_experiment,
}

architecture_name = "encoder_only"
experiment_name = "encoder_only_baseline"

data_config = SequenceDataConfig(
    data_dir=NOTEBOOK_DIR.parent / "data",
    window_size=60,
    asset_return_columns=("SPY", "TLT", "GLD", "UUP", "HYG", "LQD"),
    include_vix_log_return=True,
    include_curve_slope_change=True,
    strict_feature_availability=False,
)

encoder_only_model_config = EncoderOnlyTransformerConfig(
    d_model=64,
    embedding_dim=64,
    n_heads=4,
    num_layers=2,
    dropout=0.10,
    feedforward_multiplier=2.0,
    pooling="attention",
    use_mask_embedding=True,
)

recurrent_model_config = RecurrentBaselineConfig(
    architecture=architecture_name if architecture_name != "encoder_only" else "pure_rnn",
    input_projection_dim=32,
    rnn_hidden_dim=64,
    rnn_num_layers=2,
    lstm_hidden_dim=64,
    lstm_num_layers=1,
    embedding_dim=64,
    dropout=0.10,
    pooling="attention",
    nonlinearity="tanh",
    bidirectional=False,
    use_mask_embedding=True,
)

model_config = encoder_only_model_config if architecture_name == "encoder_only" else recurrent_model_config

training_config = TrainingConfig(
    mask_ratio=0.20,
    batch_size=64,
    num_epochs=15,
    learning_rate=1e-3,
    weight_decay=1e-4,
    train_ratio=0.8,
    early_stopping_patience=5,
    min_epochs=5,
    random_state=42,
    log_every=1,
    device="auto",
)

clustering_config = ClusteringConfig(
    target_cluster_count=None,
    cluster_candidates=(2, 3, 4, 5, 6),
    n_init=20,
    batch_size=256,
)

hmm_config = HMMReferenceConfig(
    enabled=HMMLEARN_AVAILABLE,
    state_candidates=(2, 3, 4),
    covariance_type="diag",
    n_restarts=6,
    n_iter=500,
)

print("architecture_name =", architecture_name)
print("experiment_name   =", experiment_name)
print("hmm_enabled       =", hmm_config.enabled)


## Run One Experiment

In [ ]:
runner = ARCHITECTURE_RUNNERS[architecture_name]

single_result = runner(
    experiment_name=experiment_name,
    data_config=data_config,
    model_config=model_config,
    training_config=training_config,
    clustering_config=clustering_config,
    hmm_config=hmm_config,
)

display(compare_experiment_summaries([single_result]))
display(single_result["history_df"].tail())
display(single_result["cluster_scan"])
display(single_result["cluster_summary"])

if single_result["comparison_table"] is not None:
    display(single_result["comparison_table"])

if single_result["hmm_results"] is not None:
    display(single_result["hmm_results"]["bic_summary"])


## Batch Comparison Template

这里放一组实验设置。现在先给两个 encoder-only 基线，后面你可以继续加别的架构，只要把 `architecture` 和对应配置补进去即可。

In [ ]:
experiment_setups = [
    {
        "name": "encoder_only_small",
        "architecture": "encoder_only",
        "data_config": SequenceDataConfig(
            data_dir=NOTEBOOK_DIR.parent / "data",
            window_size=60,
            asset_return_columns=("SPY", "TLT", "GLD", "UUP", "HYG", "LQD"),
            include_vix_log_return=True,
            include_curve_slope_change=True,
            strict_feature_availability=False,
        ),
        "model_config": EncoderOnlyTransformerConfig(
            d_model=64,
            embedding_dim=32,
            n_heads=4,
            num_layers=2,
            dropout=0.10,
            feedforward_multiplier=2.0,
            pooling="attention",
            use_mask_embedding=True,
        ),
        "training_config": TrainingConfig(
            mask_ratio=0.20,
            batch_size=64,
            num_epochs=12,
            learning_rate=1e-3,
            weight_decay=1e-4,
            train_ratio=0.8,
            early_stopping_patience=4,
            min_epochs=5,
            random_state=42,
            log_every=1,
            device="auto",
        ),
        "clustering_config": ClusteringConfig(target_cluster_count=None),
        "hmm_config": HMMReferenceConfig(enabled=HMMLEARN_AVAILABLE),
    },
    {
        "name": "encoder_only_wider",
        "architecture": "encoder_only",
        "data_config": SequenceDataConfig(
            data_dir=NOTEBOOK_DIR.parent / "data",
            window_size=60,
            asset_return_columns=("SPY", "TLT", "GLD", "UUP", "HYG", "LQD"),
            include_vix_log_return=True,
            include_curve_slope_change=True,
            strict_feature_availability=False,
        ),
        "model_config": EncoderOnlyTransformerConfig(
            d_model=128,
            embedding_dim=64,
            n_heads=4,
            num_layers=3,
            dropout=0.10,
            feedforward_multiplier=2.0,
            pooling="attention",
            use_mask_embedding=True,
        ),
        "training_config": TrainingConfig(
            mask_ratio=0.20,
            batch_size=64,
            num_epochs=12,
            learning_rate=8e-4,
            weight_decay=1e-4,
            train_ratio=0.8,
            early_stopping_patience=4,
            min_epochs=5,
            random_state=42,
            log_every=1,
            device="auto",
        ),
        "clustering_config": ClusteringConfig(target_cluster_count=None),
        "hmm_config": HMMReferenceConfig(enabled=HMMLEARN_AVAILABLE),
    },
    {
        "name": "pure_rnn_baseline",
        "architecture": "pure_rnn",
        "data_config": SequenceDataConfig(
            data_dir=NOTEBOOK_DIR.parent / "data",
            window_size=60,
            asset_return_columns=("SPY", "TLT", "GLD", "UUP", "HYG", "LQD"),
            include_vix_log_return=True,
            include_curve_slope_change=True,
            strict_feature_availability=False,
        ),
        "model_config": RecurrentBaselineConfig(
            architecture="pure_rnn",
            input_projection_dim=32,
            rnn_hidden_dim=64,
            rnn_num_layers=2,
            lstm_hidden_dim=64,
            lstm_num_layers=1,
            embedding_dim=64,
            dropout=0.10,
            pooling="attention",
            nonlinearity="tanh",
            bidirectional=False,
            use_mask_embedding=True,
        ),
        "training_config": TrainingConfig(
            mask_ratio=0.20,
            batch_size=64,
            num_epochs=12,
            learning_rate=1e-3,
            weight_decay=1e-4,
            train_ratio=0.8,
            early_stopping_patience=4,
            min_epochs=5,
            random_state=42,
            log_every=1,
            device="auto",
        ),
        "clustering_config": ClusteringConfig(target_cluster_count=None),
        "hmm_config": HMMReferenceConfig(enabled=HMMLEARN_AVAILABLE),
    },
    {
        "name": "rnn_lstm_hybrid_baseline",
        "architecture": "rnn_lstm_hybrid",
        "data_config": SequenceDataConfig(
            data_dir=NOTEBOOK_DIR.parent / "data",
            window_size=60,
            asset_return_columns=("SPY", "TLT", "GLD", "UUP", "HYG", "LQD"),
            include_vix_log_return=True,
            include_curve_slope_change=True,
            strict_feature_availability=False,
        ),
        "model_config": RecurrentBaselineConfig(
            architecture="rnn_lstm_hybrid",
            input_projection_dim=32,
            rnn_hidden_dim=64,
            rnn_num_layers=1,
            lstm_hidden_dim=64,
            lstm_num_layers=1,
            embedding_dim=64,
            dropout=0.10,
            pooling="attention",
            nonlinearity="tanh",
            bidirectional=False,
            use_mask_embedding=True,
        ),
        "training_config": TrainingConfig(
            mask_ratio=0.20,
            batch_size=64,
            num_epochs=12,
            learning_rate=1e-3,
            weight_decay=1e-4,
            train_ratio=0.8,
            early_stopping_patience=4,
            min_epochs=5,
            random_state=42,
            log_every=1,
            device="auto",
        ),
        "clustering_config": ClusteringConfig(target_cluster_count=None),
        "hmm_config": HMMReferenceConfig(enabled=HMMLEARN_AVAILABLE),
    },
]


In [ ]:
batch_results = []

for setup in experiment_setups:
    runner = ARCHITECTURE_RUNNERS[setup["architecture"]]
    result = runner(
        experiment_name=setup["name"],
        data_config=setup["data_config"],
        model_config=setup["model_config"],
        training_config=setup["training_config"],
        clustering_config=setup["clustering_config"],
        hmm_config=setup["hmm_config"],
    )
    batch_results.append(result)

comparison_df = compare_experiment_summaries(batch_results)
display(comparison_df)


## Next Step

等你后面实现新的架构时：

1. 新建一个和 `run_encoder_only_experiment(...)` 类似签名的 runner。
2. 把它注册进 `ARCHITECTURE_RUNNERS`。
3. 在 `experiment_setups` 里加对应实验配置。

这样这个 notebook 就能继续当统一的对比面板。